# W3.1 — Cosine baseline validation

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** W2.2 split, W2.3 audit pass, cosine_baseline.json.

Cosine không học tham số. Score liên tục giữ thông tin xếp hạng cho ROC-AUC/AP.
Các phép thử toy chạy trước dữ liệu thật. Không dùng nhãn node làm feature.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
import numpy as np
import torch
from utils.experiment import (
    read_json,
    write_json,
    seed_everything,
    array_hash,
    mapping_hash,
    environment,
    new_run,
    publish_result,
    expect_error,
)
from utils.link_graph import (
    validate_edges,
    canonicalize,
    bidirectional,
    with_self_loops,
    neighborhoods,
    pair_set,
    load_cora,
    make_split,
    audit_split,
)
from utils.link_metrics import (
    lp_metrics,
    cosine_scores,
)


D:\gnn-node-link-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\gnn-node-link-prediction\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
config = read_json(PROJECT_ROOT/'configs/cosine_baseline.json')
print(config)

{'split_id': 'cora_lp_v1', 'seed': 42, 'feature_preprocessing': 'raw_float64', 'epsilon': 1e-12, 'chunk_size': 256, 'evaluation_split': 'validation', 'atol': 1e-12, 'rtol': 1e-10}


In [ ]:
toy_x = np.array([[1.,0.],[1.,0.],[0.,1.],[0.,0.]])
toy_pairs = np.array([[0,0,0,1],[1,2,3,0]])
np.testing.assert_allclose(cosine_scores(toy_x,toy_pairs),[1,0,0,1])
assert lp_metrics([0,1],[0,1])['roc_auc'] == 1
assert lp_metrics([0,1],[1,0])['roc_auc'] == 0
assert lp_metrics([0,1],[1,0])['average_precision'] == .5
assert lp_metrics([0,1],[1,1])['roc_auc'] == .5
assert lp_metrics([0,1],[1,1])['average_precision'] == .5
checks = {'toy_cosine_and_rank_metrics':{'status':'passed','observed':[1,0,0,1]}}
checks['empty_pairs'] = expect_error(lambda:cosine_scores(toy_x,np.empty((2,0),dtype=int)))
checks['out_of_range'] = expect_error(lambda:cosine_scores(toy_x,np.array([[0],[4]])))
checks['nan_features'] = expect_error(lambda:cosine_scores(np.array([[np.nan],[1]]),np.array([[0],[1]])))
checks['inf_scores'] = expect_error(lambda:lp_metrics([0,1],[0,np.inf]))
checks['empty_metric'] = expect_error(lambda:lp_metrics([],[]))
checks['label_score_mismatch'] = expect_error(lambda:lp_metrics([0,1],[1]))
checks['single_class'] = expect_error(lambda:lp_metrics([1,1],[0,1]))

In [ ]:
from tasks.link_prediction import run_cosine_validation
# Preserve the historical notebook publication paths and original helper checks.
result = run_cosine_validation(PROJECT_ROOT, require_cached=False, checks=checks)
print(result['validation']); print('run_id:', result['run_id'])


In [ ]:
# Saved-split verification is now performed by tasks.link_prediction.load_saved_split.


In [ ]:
# Validation scoring/publication is now performed by tasks.link_prediction.run_cosine_validation.


{'roc_auc': 0.8121701918489497, 'average_precision': 0.8248011097161196, 'n_positive': 263, 'n_negative': 263}
run_id: 20260928T070135_w3_cosine_a81fd148


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. Cosine có tham số nào được học?
2. Tại sao không threshold score trước ROC-AUC?
3. AP khác diện tích PR hình thang như thế nào?
4. Tại sao chưa tính test metric?